In [1]:
import torch as th
from torch import Tensor
from collections import defaultdict
import os
import numpy as np

In [2]:
ckpt = th.load('checkpoint_best.pt', map_location='cpu', weights_only=False)
model_state_dict = ckpt['model']
print("Loaded checkpoint with {} parameters".format(len(model_state_dict)))

Loaded checkpoint with 937 parameters


In [3]:
for key in model_state_dict.keys():
    print(key)

encoder.sentence_encoder.embed_tokens.weight
encoder.sentence_encoder.embed_tokens.weight_scaling_factor
encoder.sentence_encoder.embed_tokens.weight_integer
encoder.sentence_encoder.embed_positions.weight
encoder.sentence_encoder.embed_positions.weight_scaling_factor
encoder.sentence_encoder.embed_positions.weight_integer
encoder.sentence_encoder.embed_positions_act.x_min
encoder.sentence_encoder.embed_positions_act.x_max
encoder.sentence_encoder.embed_positions_act.act_scaling_factor
encoder.sentence_encoder.layers.0.activation_fn_approx.input_scaling_factor
encoder.sentence_encoder.layers.0.input_act.x_min
encoder.sentence_encoder.layers.0.input_act.x_max
encoder.sentence_encoder.layers.0.input_act.act_scaling_factor
encoder.sentence_encoder.layers.0.self_attn.k_proj.weight
encoder.sentence_encoder.layers.0.self_attn.k_proj.bias
encoder.sentence_encoder.layers.0.self_attn.k_proj.fc_scaling_factor
encoder.sentence_encoder.layers.0.self_attn.k_proj.weight_integer
encoder.sentence_enco

In [4]:
extracted_info = defaultdict(dict)
quant_param_suffixes = (
    '.weight_integer',
    # '.bias_integer',
    '.fc_scaling_factor',
    '.shift',
    '.act_scaling_factor',  # including x_min, x_max
    # '.embed_scaling_factor',
)
quantized_modules = []
for key in model_state_dict.keys():
    if key.endswith(quant_param_suffixes):
        base_module_name = key.rsplit('.', 1)[0]
        if base_module_name not in quantized_modules:
            quantized_modules.append(base_module_name)
            print(f"Found quantized module: {base_module_name}")
print("Found {} quantized modules".format(len(quantized_modules)))

Found quantized module: encoder.sentence_encoder.embed_tokens
Found quantized module: encoder.sentence_encoder.embed_positions
Found quantized module: encoder.sentence_encoder.embed_positions_act
Found quantized module: encoder.sentence_encoder.layers.0.input_act
Found quantized module: encoder.sentence_encoder.layers.0.self_attn.k_proj
Found quantized module: encoder.sentence_encoder.layers.0.self_attn.v_proj
Found quantized module: encoder.sentence_encoder.layers.0.self_attn.q_proj
Found quantized module: encoder.sentence_encoder.layers.0.self_attn.k_proj_act
Found quantized module: encoder.sentence_encoder.layers.0.self_attn.v_proj_act
Found quantized module: encoder.sentence_encoder.layers.0.self_attn.q_proj_act
Found quantized module: encoder.sentence_encoder.layers.0.self_attn.softmax.act
Found quantized module: encoder.sentence_encoder.layers.0.self_attn.attn_probs_act
Found quantized module: encoder.sentence_encoder.layers.0.self_attn.attn_act
Found quantized module: encoder.se

In [5]:
param_types_to_extract = [
    'weight',
    'bias',
    'weight_integer',
    'bias_integer',
    'fc_scaling_factor',  # per-channel
    'shift',
    'act_scaling_factor',  # per-tensor
]
for base_module_name in quantized_modules:
    if 'embed' in base_module_name:
        print(f"Skipping embedding module: {base_module_name}")
        continue
    
    found_params = []
    for param_type in param_types_to_extract:
        key = f"{base_module_name}.{param_type}"
        if key in model_state_dict:
            tensor = model_state_dict[key]
            extracted_info[base_module_name][param_type] = tensor
            found_params.append(f"{param_type} (shape: {tensor.shape})")
    if found_params:
        print(f"Found parameters for {base_module_name}:")
        for param_info in found_params:
            print(f" - {param_info}")
    else:
        print(f"No parameters found for {base_module_name}")

Skipping embedding module: encoder.sentence_encoder.embed_tokens
Skipping embedding module: encoder.sentence_encoder.embed_positions
Skipping embedding module: encoder.sentence_encoder.embed_positions_act
Found parameters for encoder.sentence_encoder.layers.0.input_act:
 - act_scaling_factor (shape: torch.Size([1]))
Found parameters for encoder.sentence_encoder.layers.0.self_attn.k_proj:
 - weight (shape: torch.Size([768, 768]))
 - bias (shape: torch.Size([768]))
 - weight_integer (shape: torch.Size([768, 768]))
 - bias_integer (shape: torch.Size([768]))
 - fc_scaling_factor (shape: torch.Size([768]))
Found parameters for encoder.sentence_encoder.layers.0.self_attn.v_proj:
 - weight (shape: torch.Size([768, 768]))
 - bias (shape: torch.Size([768]))
 - weight_integer (shape: torch.Size([768, 768]))
 - bias_integer (shape: torch.Size([768]))
 - fc_scaling_factor (shape: torch.Size([768]))
Found parameters for encoder.sentence_encoder.layers.0.self_attn.q_proj:
 - weight (shape: torch.Siz

In [27]:
for key in extracted_info['encoder.sentence_encoder.embed_tokens']:
    print(f"Key: {key}, First 5 elements: {extracted_info['encoder.sentence_encoder.embed_tokens'][key][:5]}")

## Validation

In [38]:
first_linear_layer_name = None
for module_name in extracted_info:
    if 'fc_scaling_factor' in extracted_info[module_name]:
        first_linear_layer_name = module_name
        break
    
if first_linear_layer_name:
    print(f"First linear layer with 'fc_scaling_factor': {first_linear_layer_name}")
    layer_data = extracted_info[first_linear_layer_name]

    if 'weight' in layer_data and 'weight_integer' in layer_data and 'fc_scaling_factor' in layer_data:
        original_weight = layer_data['weight']
        weight_int = layer_data['weight_integer']
        scaling_factor = layer_data['fc_scaling_factor']
        
        if scaling_factor.dim() == 1:
            scaling_factor_reshaped = scaling_factor.unsqueeze(1)
        else:
            scaling_factor_reshaped = scaling_factor
            
        reconstructed_weight = weight_int.float() * scaling_factor_reshaped

        print(f"Original weight tensor (first 5 elements): {original_weight[0, :5]}")
        print(f"Reconstructed weight tensor (first 5 elements): {reconstructed_weight[0, :5]}")
        
        diff = th.mean(th.abs(original_weight - reconstructed_weight)).item()
        print(f"Mean absolute difference between original and reconstructed weights: {diff}")
    else:
        print(f"Missing required parameters to reconstruct weights for layer {first_linear_layer_name}.")
else:
    print("No linear layer with 'fc_scaling_factor' found.")

First linear layer with 'fc_scaling_factor': encoder.sentence_encoder.layers.0.self_attn.k_proj
Original weight tensor (first 5 elements): tensor([ 0.0854,  0.0397, -0.1005,  0.0129,  0.0020])
Reconstructed weight tensor (first 5 elements): tensor([ 0.0859,  0.0406, -0.1002,  0.0119,  0.0024])
Mean absolute difference between original and reconstructed weights: 0.0007616053917445242


## Transform Pattern

In [11]:
def decimal_to_hex(num: int, bit_width: int = 8) -> str:
    if num < 0:
        num = (1 << bit_width) + num
    return hex(num)[2:].zfill(2)

def save_weight_to_dat(weight_tensor: Tensor, output_dir: str, file_prefix: str, step: int = 8, bit_width: int = 8):
    """
    Converts a PyTorch weight_integer tensor to the specified .dat format and saves it.

    Args:
        weight_tensor (Tensor): The INT8 weight tensor (e.g., extracted_info[...]['weight_integer']).
        output_dir (str): The directory path to save the .dat file.
        file_prefix (str): The prefix for the .dat file (e.g., 'Wq', 'Wk', 'Wfc1').
        step (int): The step size used in the DAT file format (usually 8).
        bit_width (int): The bit width of the weights (usually 8).
    """
    os.makedirs(output_dir, exist_ok=True)

    # Convert PyTorch tensor to NumPy array (ensure it's on CPU and integer type)
    weight = weight_tensor.cpu().numpy().astype(int)
    # rows, cols = weight.shape  # error
    cols, rows = weight.shape

    output_filename = os.path.join(output_dir, f"{file_prefix}.dat")

    print(f"  > Converting weights (Shape: {weight.shape}) to {output_filename} ...")

    with open(output_filename, "w") as dat_file:
        for offset in range(step):
            for col in range(offset, cols, step):
                hex_values = []
                for row in range(rows):
                    # Ensure the value is within INT8 range (-128 to 127)
                    val = weight[col][row]
                    if not (-128 <= val <= 127):
                         print(f"    Warning: Value {val} at ({row}, {col}) is out of INT8 range! May cause conversion errors.")
                         val = max(-128, min(127, val))

                    hex_value = decimal_to_hex(val, bit_width)
                    hex_values.append(hex_value)

                    # Write every 'step' rows or when reaching the last row
                    if (row + 1) % step == 0 or row == rows - 1:
                        # Calculate the row range for the current write
                        start_row = row - len(hex_values) + 1
                        dat_file.write("".join(hex_values) + f"  // col_{col}, row_{start_row}~row_{row}\n")
                        hex_values.clear()

    print(f"  > Finished: {output_filename}")


In [12]:
base_output_dir = "params_result"
print(f"Saving parameter files to base directory: '{base_output_dir}'\n")

# Dictionary to map module types to file prefixes (for .dat file)
prefix_map = {
    'k_proj': 'Wk',
    'q_proj': 'Wq',
    'v_proj': 'Wv',
    'out_proj': 'Wo',
    'fc1': 'Wfc1',
    'fc2': 'Wfc2',
}

param_types_to_save_npy = [
    'weight',             # Original FP32 weights
    'bias',               # Bias terms
    'weight_integer',    # INT8 weights
    'fc_scaling_factor',  # Scaling factors for linear layers
    'shift',              # Shift values for quantization
    'act_scaling_factor', # Scaling factors for activations
]

unique_quantized_modules = sorted(list(set(quantized_modules)))
processed_modules_count = 0
saved_files_count = 0

for module_name in unique_quantized_modules:
    if module_name in extracted_info:
        module_data = extracted_info[module_name]
        print(f"--- Processing module: {module_name} ---")
        processed_modules_count += 1

        relative_path = module_name.replace('.', os.sep)
        output_dir = os.path.join(base_output_dir, relative_path)
        os.makedirs(output_dir, exist_ok=True)

        # ==== Save weight_integer as .dat ====
        if 'weight_integer' in module_data:
            weight_int_tensor = module_data['weight_integer']
            module_suffix = module_name.split('.')[-1]
            file_prefix = prefix_map.get(module_suffix, f"W_{module_suffix}")
            try:
                save_weight_to_dat(weight_int_tensor, output_dir, file_prefix)
                saved_files_count += 1
            except Exception as e:
                print(f"  Error saving weight_integer for {module_name} as .dat: {e}")

        # ==== Save other specified parameters as .npy ====
        for param_type in param_types_to_save_npy:
            if param_type in module_data:
                tensor = module_data[param_type]
                numpy_arr = tensor.cpu().numpy()
                npy_filename = os.path.join(output_dir, f"{param_type}.npy")
                
                try:
                    np.save(npy_filename, numpy_arr)
                    print(f"  > Saved: {npy_filename} (Shape: {numpy_arr.shape})")
                    saved_files_count += 1
                except Exception as e:
                    print(f"  Error saving {param_type} for {module_name} as .npy: {e}")


print(f"Checked a total of {len(quantized_modules)} quantized module names.")
print(f"Processed {processed_modules_count} modules found in extracted_info.")
print(f"Successfully saved a total of {saved_files_count} parameter files (.dat and .npy).")
print(f"Files saved under the '{base_output_dir}' directory in corresponding subdirectories.")

Saving parameter files to base directory: 'params_result'

--- Processing module: encoder.sentence_encoder.emb_layer_norm ---
  > Saved: params_result\encoder\sentence_encoder\emb_layer_norm\weight.npy (Shape: (768,))
  > Saved: params_result\encoder\sentence_encoder\emb_layer_norm\bias.npy (Shape: (768,))
  > Saved: params_result\encoder\sentence_encoder\emb_layer_norm\shift.npy (Shape: (1,))
--- Processing module: encoder.sentence_encoder.emb_layer_norm.activation ---
  > Saved: params_result\encoder\sentence_encoder\emb_layer_norm\activation\act_scaling_factor.npy (Shape: (1,))
--- Processing module: encoder.sentence_encoder.layers.0.fc1 ---
  > Converting weights (Shape: (3072, 768)) to params_result\encoder\sentence_encoder\layers\0\fc1\Wfc1.dat ...
  > Finished: params_result\encoder\sentence_encoder\layers\0\fc1\Wfc1.dat
  > Saved: params_result\encoder\sentence_encoder\layers\0\fc1\weight.npy (Shape: (3072, 768))
  > Saved: params_result\encoder\sentence_encoder\layers\0\fc1\bi

In [40]:
validation_module_name = 'encoder.sentence_encoder.layers.11.fc1'
base_output_dir = "params_dat_result"

if validation_module_name in extracted_info:
    relative_path = validation_module_name.replace('.', os.sep)
    module_output_dir = os.path.join(base_output_dir, relative_path)
    
    weight_npy_path = os.path.join(module_output_dir, 'weight.npy')
    scale_npy_path = os.path.join(module_output_dir, 'fc_scaling_factor.npy')
    bias_npy_path = os.path.join(module_output_dir, 'bias.npy')
    
    layer_data_in_memory = extracted_info[validation_module_name]

    all_files_exist = (os.path.exists(weight_npy_path) and 
                       os.path.exists(scale_npy_path) and 
                       os.path.exists(bias_npy_path))
                       
    all_data_in_memory = ('weight_integer' in layer_data_in_memory and
                          'bias' in layer_data_in_memory)

    if all_files_exist and all_data_in_memory:
        
        try:
            loaded_original_weight_np = np.load(weight_npy_path)
            loaded_scaling_factor_np = np.load(scale_npy_path)
            loaded_bias_np = np.load(bias_npy_path) 
            
            original_weight = th.from_numpy(loaded_original_weight_np).float()
            scaling_factor = th.from_numpy(loaded_scaling_factor_np).float()
            loaded_bias = th.from_numpy(loaded_bias_np).float() # <-- 轉換 bias

            weight_int_tensor = layer_data_in_memory['weight_integer']
            bias_in_memory = layer_data_in_memory['bias'].float()

            print(f"成功讀取驗證檔案: {validation_module_name}")
            print(f" - 已讀取 'weight.npy' (Shape: {original_weight.shape})")
            print(f" - 已讀取 'fc_scaling_factor.npy' (Shape: {scaling_factor.shape})")
            print(f" - 已讀取 'bias.npy' (Shape: {loaded_bias.shape})")
            print(f" - 使用記憶體中的 'weight_integer' (Shape: {weight_int_tensor.shape})")
            print(f" - 使用記憶體中的 'bias' (Shape: {bias_in_memory.shape})")

            if scaling_factor.dim() == 1:
                scaling_factor_reshaped = scaling_factor.unsqueeze(1)
            else:
                scaling_factor_reshaped = scaling_factor
                
            reconstructed_weight = weight_int_tensor.float() * scaling_factor_reshaped

            print("\n[權重重建檢查]")
            print(f"原始 (來自 .npy) 權重 (前 5 元素): {original_weight[0, :5]}")
            print(f"重建 (來自 .npy scale) 權重 (前 5 元素): {reconstructed_weight[0, :5]}")
            
            weight_diff = th.mean(th.abs(original_weight - reconstructed_weight)).item()
            print(f"平均絕對差異值: {weight_diff}")
            
            if weight_diff < 0.001: 
                print(" -> 權重驗證通過")
            else:
                print(" -> 權重驗證失敗")

            print("\n[Bias 檢查]")
            bias_check_passed = th.allclose(loaded_bias, bias_in_memory) 
            
            print(f"從 'bias.npy' 讀取 (前 5 元素): {loaded_bias[:5]}")
            print(f"從 記憶體 讀取 (前 5 元素):    {bias_in_memory[:5]}")
            
            if bias_check_passed:
                print(" -> Bias 驗證通過 (儲存的 .npy 與記憶體中的值完全一致)")
            else:
                print(" -> Bias 驗證失敗 (值不一致)")

        except Exception as e:
            print(f"驗證過程中發生錯誤 {validation_module_name}: {e}")
            
    else:
        print(f"驗證失敗: 找不到 {validation_module_name} 所需的檔案或記憶體資料")
        if not os.path.exists(weight_npy_path):
            print(f" - 找不到檔案: {weight_npy_path}")
        if not os.path.exists(scale_npy_path):
            print(f" - 找不到檔案: {scale_npy_path}")
        if not os.path.exists(bias_npy_path):
            print(f" - 找不到檔案: {bias_npy_path}")
        if 'weight_integer' not in layer_data_in_memory:
            print(" - 找不到: 'weight_integer' (在記憶體中)")
        if 'bias' not in layer_data_in_memory:
            print(" - 找不到: 'bias' (在記憶體中)")
else:
    print(f"驗證失敗: 模組 '{validation_module_name}' 不在 extracted_info 中。")

成功讀取驗證檔案: encoder.sentence_encoder.layers.11.fc1
 - 已讀取 'weight.npy' (Shape: torch.Size([3072, 768]))
 - 已讀取 'fc_scaling_factor.npy' (Shape: torch.Size([3072]))
 - 已讀取 'bias.npy' (Shape: torch.Size([3072]))
 - 使用記憶體中的 'weight_integer' (Shape: torch.Size([3072, 768]))
 - 使用記憶體中的 'bias' (Shape: torch.Size([3072]))

[權重重建檢查]
原始 (來自 .npy) 權重 (前 5 元素): tensor([-0.1131, -0.1521, -0.0664, -0.0238, -0.0281])
重建 (來自 .npy scale) 權重 (前 5 元素): tensor([-0.1132, -0.1516, -0.0658, -0.0237, -0.0274])
平均絕對差異值: 0.00031008265796117485
 -> 權重驗證通過

[Bias 檢查]
從 'bias.npy' 讀取 (前 5 元素): tensor([-0.2150, -0.0909, -0.0697, -0.0467, -0.0835])
從 記憶體 讀取 (前 5 元素):    tensor([-0.2150, -0.0909, -0.0697, -0.0467, -0.0835])
 -> Bias 驗證通過 (儲存的 .npy 與記憶體中的值完全一致)
